# 18 — Censoring-Aware Growth Audit (v2.5)

**LeukemiPrediag v2.5 — outcome-free, research use only.**

This notebook evaluates whether recovering sub-1% / zero-VAF measurements and using a simple censoring-aware observation model improves one-step-ahead VAF forecasting.

Pre-specified rules:

- do not modify `src/temporal_validation.py` or `src/growth_shrinkage.py`;
- patient-held-out 5-fold cross-fitting, seed 17;
- repeat over 20 CV seeds;
- leave-one-cohort-out transport;
- patient-cluster bootstrap confidence intervals;
- compare against `carry_forward`, `global_alpha`, `drift_global`, and the frozen exploratory 2% `alpha_vaf_stratum`;
- accept the new point forecaster only if it beats both `alpha_vaf_stratum` and `drift_global` with bootstrap CIs excluding 0 and does not worsen under LOCO.

This validates **VAF forecasting only**. It does not validate leukemia outcomes.


In [ ]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    MYDRIVE = Path("/content/drive/MyDrive")
    candidates = [
        MYDRIVE / "leukemia-prediagnose" / "blood-cancer-early-detection",
        MYDRIVE / "blood-cancer-early-detection",
    ]
    candidates += list(MYDRIVE.glob("*/blood-cancer-early-detection"))
except ImportError:
    candidates = [Path.cwd(), Path.cwd().parent]

PROJECT_ROOT = next((p for p in candidates if (p / "src" / "growth_censored.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing src/growth_censored.py")

os.chdir(PROJECT_ROOT)
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

UNFILTERED = PROJECT_ROOT / "data" / "processed" / "v2" / "real_serial_unfiltered.csv"
FILTERED = PROJECT_ROOT / "data" / "processed" / "v2" / "real_serial_filtered.csv"
OUTDIR = PROJECT_ROOT / "data" / "processed" / "v2" / "growth_censored_v25"
OUTDIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("unfiltered exists:", UNFILTERED.exists())
print("filtered exists:", FILTERED.exists())


In [ ]:
from growth_censored import (
    build_trajectory_table, add_censored_candidates,
    crossfit_all, leave_one_cohort_out, errors,
    cluster_bootstrap_delta, MODELS,
)
from growth_shrinkage import build_trajectory_table as build_filtered_trajectory_table
from interval_calibration import evaluate_interval_calibration

serial = pd.read_csv(UNFILTERED)
print("rows:", len(serial))
print("patients:", serial.patient_id.nunique())
print("trajectories:", serial.groupby(["patient_id", "variant_id"]).ngroups)
print("sub-1% rows:", int((serial.vaf < 0.01).sum()))
print("zero-VAF rows:", int((serial.vaf == 0).sum()))

tab = build_trajectory_table(serial)
tab = add_censored_candidates(tab)

print("eligible trajectories:", len(tab))
print("eligible patients:", tab.patient_id.nunique())
display(tab.groupby("cohort").size().rename("eligible_trajectories"))


## A. Primary patient-held-out validation

The censoring-aware trajectory fit uses only that trajectory's **training timepoints**. Cross-patient parameters for the simple baselines are fitted inside each outer training fold.


In [ ]:
P, params = crossfit_all(tab, n_splits=5, seed=17)
E = errors(P, tab)

primary = E.mean().sort_values()
display(primary.rename("mean_abs_log_vaf_error").to_frame())

by_cohort = E.groupby(tab.cohort).mean()
display(by_cohort)

boot = {}
for ref in ["carry_forward", "global_alpha", "drift_global", "alpha_vaf_stratum"]:
    boot[f"censored_growth_minus_{ref}"] = cluster_bootstrap_delta(
        E["censored_growth"], E[ref], tab.patient_id, n_boot=4000, seed=0
    )
display(pd.DataFrame(
    boot, index=["delta", "ci_low", "ci_high", "p_new_better"]
).T)


## B. Repeated CV + leave-one-cohort-out

No hyperparameter or threshold is tuned to improve these results. The v2.4 2% VAF split is kept fixed and remains exploratory.


In [ ]:
rep = []
for seed in range(20):
    Ps, _ = crossfit_all(tab, n_splits=5, seed=seed)
    rep.append(errors(Ps, tab).mean())
rep = pd.DataFrame(rep)

repeated_summary = pd.DataFrame({
    "mean": rep.mean(),
    "min": rep.min(),
    "max": rep.max(),
}).sort_values("mean")
display(repeated_summary)

LP = leave_one_cohort_out(tab)
LE = errors(LP, tab)
loco_overall = LE.mean().sort_values()
loco_by_cohort = LE.groupby(tab.cohort).mean()

print("LOCO overall")
display(loco_overall.rename("mean_abs_log_vaf_error").to_frame())
print("LOCO by cohort")
display(loco_by_cohort)


In [ ]:
b_alpha = boot["censored_growth_minus_alpha_vaf_stratum"]
b_drift = boot["censored_growth_minus_drift_global"]

beats_alpha = b_alpha[2] < 0
beats_drift = b_drift[2] < 0
loco_ok = (
    loco_overall["censored_growth"] <= loco_overall["alpha_vaf_stratum"]
    and loco_overall["censored_growth"] <= loco_overall["drift_global"]
)
accepted = bool(beats_alpha and beats_drift and loco_ok)

decision = {
    "beats_alpha_vaf_stratum_ci_excludes_0": bool(beats_alpha),
    "beats_drift_global_ci_excludes_0": bool(beats_drift),
    "not_worse_under_loco_vs_both": bool(loco_ok),
    "accepted_as_core_point_forecaster": accepted,
}
print(json.dumps(decision, indent=2))


## C. Cohort-aware 80% interval calibration

Point forecasts are unchanged. Cohort-specific residual scales are learned only from training patients; when training support is insufficient, calibration falls back to the pooled training residual scale.


In [ ]:
filtered = pd.read_csv(FILTERED)
filtered_tab = build_filtered_trajectory_table(filtered)

interval_pred, interval_overall, interval_by_cohort = evaluate_interval_calibration(
    filtered_tab,
    n_splits=5,
    seed=17,
    target_coverage=0.80,
    min_cohort_cal=20,
)

display(interval_overall)
display(interval_by_cohort)

interval_pred.to_csv(OUTDIR / "v25_interval_filtered_predictions.csv", index=False)
interval_overall.to_csv(OUTDIR / "v25_interval_filtered_overall.csv", index=False)
interval_by_cohort.to_csv(OUTDIR / "v25_interval_filtered_by_cohort.csv", index=False)


In [ ]:
# Save reproducibility artifacts.
save_cols = [
    "patient_id", "cohort", "variant_id", "gene", "n_train",
    "horizon", "last_vaf", "true_vaf", "s_ols",
]
forecast_out = pd.concat([
    tab[save_cols],
    P[MODELS].add_prefix("logpred_"),
    E.add_prefix("abs_log_err_"),
], axis=1)

forecast_out.to_csv(OUTDIR / "v25_crossfit_forecasts.csv", index=False)
rep.to_csv(OUTDIR / "v25_repeated_cv_20seeds.csv", index=False)
by_cohort.reset_index().to_csv(OUTDIR / "v25_primary_by_cohort.csv", index=False)
loco_by_cohort.reset_index().to_csv(OUTDIR / "v25_loco_by_cohort.csv", index=False)

audit = {
    "version": "LeukemiPrediag-v2.5-censoring-aware-audit",
    "research_use_only": True,
    "not_a_clinical_outcome_validation": True,
    "input": "data/processed/v2/real_serial_unfiltered.csv",
    "n_eligible_trajectories": int(len(tab)),
    "n_patients": int(tab.patient_id.nunique()),
    "censored_model": {
        "observation": "binomial for LBC; left-censored Gaussian on logit VAF for SardiNIA",
        "fixed_slope_prior_sd_per_year": 0.1,
        "ridge_lambda": 50.0,
    },
    "primary_seed17_mae_log": {k: round(float(v), 5) for k, v in E.mean().items()},
    "primary_params": params,
    "repeated_cv_20seeds": {
        m: [round(float(rep[m].mean()), 5),
            round(float(rep[m].min()), 5),
            round(float(rep[m].max()), 5)]
        for m in MODELS
    },
    "loco_overall": {k: round(float(v), 5) for k, v in LE.mean().items()},
    "bootstrap_delta_mean_lo_hi_pbetter": {
        k: [round(float(x), 5) for x in v] for k, v in boot.items()
    },
    "acceptance": decision,
    "primary_by_cohort": by_cohort.round(5).to_dict(orient="index"),
    "loco_by_cohort": loco_by_cohort.round(5).to_dict(orient="index"),
    "interval_calibration": {
        "overall": interval_overall.to_dict(orient="records"),
        "by_cohort": interval_by_cohort.to_dict(orient="records"),
    },
}
with open(OUTDIR / "v25_censored_growth_audit.json", "w") as f:
    json.dump(audit, f, indent=2, default=float)

print("Saved to:", OUTDIR)


## Interpretation boundary

A failure of the simple censoring-aware **point forecaster** does not imply that sub-floor measurements are unimportant. It means the tested latent-linear observation model does not satisfy the pre-specified generalization criterion across these heterogeneous cohorts.

Cohort-aware conformal scaling may still be useful for **research uncertainty reporting** if it improves per-cohort coverage without using held-out-patient residuals for calibration.

No result in this notebook is a leukemia-risk or clinical-outcome result.
